In [ ]:
from pydantic import BaseModel
from enum import Enum
import torch

class RandomType(Enum):
    uniform = "uniform"
    loguniform = "loguniform"

class RandomnessConfig(BaseModel):
    random_type: RandomType = RandomType.uniform
    min_value: float
    max_value: float

class RandomModifierConfig(BaseModel):
    additive: RandomnessConfig = RandomnessConfig(random_type=RandomType.uniform, min_value=0., max_value=0.)
    multiplicative: RandomnessConfig = RandomnessConfig(random_type=RandomType.uniform, min_value=1., max_value=1.)
    # multiplicative_first: bool = True

In [19]:
modifier_config = RandomModifierConfig(multiplicative=RandomnessConfig(random_type=RandomType.loguniform, min_value=1e-5, max_value=1.))

In [26]:
def random_tensor_uniform(min_value, max_value, shape, generator=None, device="cpu"):
    return min_value + torch.rand(shape, generator=generator, device=device) * (max_value - min_value)

def random_tensor_loguniform(min_value, max_value, shape, generator=None, device="cpu"):
    return min_value * torch.exp(torch.rand(shape, generator=generator, device=device) * (torch.log(torch.tensor(max_value)) - torch.log(torch.tensor(min_value))))
def random_tensor(config: RandomnessConfig, shape, generator=None, device="cpu"):
    if config.random_type == RandomType.uniform:
        return random_tensor_uniform(config.min_value, config.max_value, shape, generator=generator, device=device)
    elif config.random_type == RandomType.loguniform:
        return random_tensor_loguniform(config.min_value, config.max_value, shape, generator=generator, device=device)
    else:
        raise ValueError(f"Unknown random type: {config.random_type}")

In [27]:
random_tensor(modifier_config.multiplicative, (5,)), random_tensor(modifier_config.additive, (5,), device="cuda")


(tensor([2.7591e-05, 7.0546e-05, 5.3528e-02, 7.0250e-02, 8.9547e-01]),
 tensor([0., 0., 0., 0., 0.], device='cuda:0'))

In [28]:
def apply_randomness(value: torch.Tensor, config: RandomModifierConfig, generator=None) -> torch.Tensor:
    # if config.multiplicative_first:
    mult = random_tensor(config.multiplicative, value.shape, device=value.device, generator=generator)
    add = random_tensor(config.additive, value.shape, device=value.device, generator=generator)
    return value*mult + add


In [34]:
apply_randomness(torch.randint(5, (5,5)), modifier_config)

tensor([[2.2636e-02, 5.8365e-02, 1.2935e-04, 2.3098e-04, 3.0860e-01],
        [0.0000e+00, 8.3803e-02, 0.0000e+00, 7.8931e-04, 6.0625e-03],
        [0.0000e+00, 5.0985e-01, 2.4441e-01, 4.1126e-02, 2.6765e-02],
        [4.4438e-05, 5.6373e-02, 5.4024e-03, 1.9064e+00, 2.7252e-01],
        [0.0000e+00, 6.3152e-01, 6.7725e-05, 3.8332e-01, 3.8395e-04]])